# tSCS EMG — 30 Hz burst vs ARC-EX (subject NTA, 24-07-2026)

Does the **stimulation mode** matter? The same session gives both modes in all four
polarity × lidocaine conditions, so burst and ARC-EX can be compared *within* each condition:
**burst at `BURST_MA`** (35 mA) vs **ARC-EX at `ARCEX_MA`** (110 mA) — roughly matched relative to
their motor thresholds (burst 25–30, ARC-EX 65–90 mA).

**Colours / styles:** **black = 30 Hz burst, red = ARC-EX** (mode); **solid = cathodic, dashed /
hatched = anodic** (polarity). Before and with lidocaine are separate figures.

## What is compared
For each pulse (burst) or packet (ARC-EX): `p2p = max − min` in `[onset + RESP_START_MS, next
onset − GUARD_MS]`; same criterion and artifact rules as the other notebooks. The train structure
is identical (30 Hz, 33.4 ms), so the per-pulse figures line up.

## Files
| | burst | ARC-EX |
|---|---|---|
| cathodic · before | `102443` | `103530` |
| cathodic · lidocaine | `113834` | `114332` |
| anodic · before | `102721` | `103849` |
| anodic · lidocaine | `114055` | `114730` |

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from functions import set_style, load_run, pretty
from functions.burst import compare_at_intensity, summary_curves, plot_pulse_overlay
from functions.paper import fig_train_modes
set_style()


## 1 · Config

In [ ]:
D = "tSCS_CHUV_data/24-07-2026/testSCS/"
BURST_MA, ARCEX_MA = 35, 110          # <-- the intensities compared

FILES = {   # condition: (burst file, ARC-EX file)
    "cathodic · before":    ("Burst_autosave_20260724_102443_670ms.csv", "Modulated_autosave_20260724_103530_508ms.csv"),
    "cathodic · lidocaine": ("Burst_autosave_20260724_113834_522ms.csv", "Modulated_autosave_20260724_114332_473ms.csv"),
    "anodic · before":      ("Burst_autosave_20260724_102721_694ms.csv", "Modulated_autosave_20260724_103849_459ms.csv"),
    "anodic · lidocaine":   ("Burst_autosave_20260724_114055_891ms.csv", "Modulated_autosave_20260724_114730_561ms.csv"),
}
MODE_COL = {"burst": "black", "arcex": "#d62728"}
POL_STYLE = {"cathodic": ("", "-", "o"), "anodic": ("///", "--", "s")}     # hatch, line style, marker

N_PULSES, RESP_START_MS, GUARD_MS, MIN_SNR, MAX_EDGE_FRAC = 10, 8.0, 1.0, 2.0, 0.5
KW = dict(n_pulses=N_PULSES, resp_start_ms=RESP_START_MS, guard_ms=GUARD_MS, min_snr=MIN_SNR,
          max_edge_frac=MAX_EDGE_FRAC)

meta0, _, sig0 = load_run(D + FILES["cathodic · before"][0])
muscles = [c for c in sig0 if c != "Trigger A"]
for k, (b, a) in FILES.items():
    mb, _, _ = load_run(D + b); ma, _, _ = load_run(D + a)
    print(f"{k:22s} burst {BURST_MA} mA {'ok' if BURST_MA in [m['amp_ma'] for m in mb] else 'MISSING'} | "
          f"ARC-EX {ARCEX_MA} mA {'ok' if ARCEX_MA in [m['amp_ma'] for m in ma] else 'MISSING'}")

def pair(cond):     # (files, labels, colours, hatches, linestyles) for burst vs ARC-EX of one condition
    h, ls, _ = POL_STYLE[cond.split(" · ")[0]]
    return ([D + FILES[cond][0], D + FILES[cond][1]], [f"30 Hz burst · {cond}", f"ARC-EX · {cond}"],
            [MODE_COL["burst"], MODE_COL["arcex"]], [h, h], [ls, ls])


## 2 · Burst vs ARC-EX within each condition — all muscles

One figure per condition: traces with the detected ▼/▲ (black = burst at 35 mA, red = ARC-EX at
110 mA), per-pulse peak-to-peak in mV, pulse 1 vs mean of the rest.

In [ ]:
for cond in FILES:
    files, labs, cols, hat, ls = pair(cond)
    compare_at_intensity(files, None, amp=(BURST_MA, ARCEX_MA), normalize="none", labels=labs,
                         colours=cols, hatches=hat, linestyles=ls, title=cond, **KW)


## 3 · One muscle, all four conditions, both modes

Eight trains on one figure for the chosen muscle: black/red = mode, solid/dashed (plain/hatched)
= cathodic/anodic; before and with lidocaine as two figures.

In [ ]:
MUSCLE = "Flex. digitorum (R)"
for state in ("before", "lidocaine"):
    files, labs, cols, hat, ls = [], [], [], [], []
    for pol in ("cathodic", "anodic"):
        f_, l_, c_, h_, s_ = pair(f"{pol} · {state}")
        files += f_; labs += l_; cols += c_; hat += h_; ls += s_
    compare_at_intensity(files, None, amp=(BURST_MA, ARCEX_MA) * 2, normalize="none", muscles=MUSCLE,
                         labels=labs, colours=cols, hatches=hat, linestyles=ls, title=f"{MUSCLE} - {state}", **KW)


## 4 · Depression along the train — burst vs ARC-EX, each mode across its own intensities

Recruitment curves of burst and ARC-EX for one condition at a time (x-axes differ, so read the
bottom row — mean of pulses 2–10 as % of pulse 1 — rather than the mV).

In [ ]:
for cond in FILES:
    files, labs, cols, hat, ls = pair(cond)
    summary_curves(files, None, labels=labs, colours=cols, markers=[POL_STYLE[cond.split(" · ")[0]][2]] * 2, **KW)


## 5 · Paper-style figure — 2–3 muscles, burst vs ARC-EX (cathodic · before)

`fig_train_modes` from `functions/paper.py`: raw traces with scale bars on the left, pulse 1 vs
mean of pulses 2–10 (% of own pulse 1) on the right. Change `COND_FIG` / `MUSCLES_FIG` / `SAVE`.

In [ ]:
COND_FIG    = "cathodic · before"
MUSCLES_FIG = ["Flex. digitorum (R)", "Flex. carpi rad. (R)", "Ext. digitorum (L)"]
SAVE        = None            # e.g. "figures/burst_vs_arcex_24-07.png"

specs = [dict(label="30 Hz burst", csv=D + FILES[COND_FIG][0], amp=BURST_MA, colour="black"),
         dict(label="ARC-EX",      csv=D + FILES[COND_FIG][1], amp=ARCEX_MA, colour="#d62728")]
fig_train_modes(specs, MUSCLES_FIG, n_pulses=N_PULSES, resp_start_ms=RESP_START_MS, title=COND_FIG, save=SAVE);
